In [1]:
from pathlib import Path
from datetime import datetime
from collections import OrderedDict
from pathlib import Path
from io import StringIO
import re
import pandas as pd
import numpy as np
import scipy.constants as sc
import scipy.special   as sp
import scipy.interpolate as si
import astropy.constants as const
import sys
import math
import requests

def post_psg_config(psg_base_url: str, config_text: str, timeout: int = 120) -> str:
    url = psg_base_url.rstrip("/") + "/api.php"
    resp = requests.post(url, data={"file": config_text}, timeout=timeout)
    resp.raise_for_status()

    reply_raw = resp.text

    reply_header = []
    reply_data = []
    for line in reply_raw.splitlines():
        if line.startswith('#'):
            reply_header.append(line)
        else:
            reply_data.append(line)

    reply_header_str = '\n'.join(reply_header)
    reply_data_np = np.loadtxt(StringIO('\n'.join(reply_data)))


    return {'header': reply_header_str, 'spectrum': reply_data_np}

# --- configuration ---
PSG_URL = "https://psg.gsfc.nasa.gov/"
#PSG_URL = "http://localhost:32768/"             # local PSG Docker


def config_str_to_dict(config_str):
        psg_regex = r"<(.+?)>(.*)"
        matches = re.finditer(psg_regex, config_str, re.MULTILINE)

        ret = OrderedDict()
        for match in matches:
            key = match.group(1)
            value = match.group(2)
            try:
                value = int(value)
            except ValueError:
                try:
                    value = float(value)
                except ValueError:
                    value = value
            ret[key] = value
        return ret

def config_dict_to_str(config_dict):
        ret = []
        for key, value in config_dict.items():
            ret.append('<{}>{}'.format(key, str(value)))
        return '\n'.join(ret)

# Order matters and must match how you’ll write <ATMOSPHERE-GAS> and per-layer VMRs:
MOLECULES = ['H2O','CO2','O2','N2','CH4','N2O','CO','O3','SO2','NH3','C2H6','NO2']

HITDICT   = {'H2O':'HIT[1]','CO2':'HIT[2]','O3':'HIT[3]','N2O':'HIT[4]',
             'CO':'HIT[5]','CH4':'HIT[6]','O2':'HIT[7]','NO':'HIT[8]',
             'SO2':'HIT[9]','NO2':'HIT[10]','NH3':'HIT[11]','N2':'HIT[22]',
             'HCN':'HIT[23]','C2H6':'HIT[27]','H2S':'HIT[31]'}

molweights = np.array([18.02, 44.01, 31.9988, 28.0134, 16.043, 44.013, 
                           28.011, 47.998, 64.06, 17.02, 28.05, 46.0055]) #g/mol


def _layer_order_array(a, order="top_first"):
    """Frontier frequently writes LAYER-1 at TOP (low P)."""
    if order == "top_first":
        return a[::-1]  # top -> surface
    return a           # surface -> top

def PT_line(pressure, params, R_star, T_star, T_int, sma, grav):
  '''
  Generats a PT profile based on input free parameters and pressure array.
  If no inputs are provided, it will run in demo mode, using free
  parameters given by the Line 2013 paper and some dummy pressure
  parameters.

  Inputs
  ------
  pressure: 1D float ndarray
     Array of pressure values in bars.
  params: 1D float ndarray
     Array of free parameters:
       - log10(kappa):  Planck thermal IR opacity in units cm^2/gr
       - log10(gamma1): Visible-to-thermal stream Planck mean opacity ratio.
       - log10(gamma2): Visible-to-thermal stream Planck mean opacity ratio.
       - alpha:         Visible-stream partition (0.0--1.0).
       - beta:          A 'catch-all' for albedo, emissivity, and day-night
                      redistribution (on the order of unity).
  R_star: Float
     Stellar radius (in meters).
  T_star: Float
     Stellar effective temperature (in Kelvin degrees).
  T_int:  Float
     Planetary internal heat flux (in Kelvin degrees).
  sma:    Float
     Semi-major axis (in meters).
  grav:   Float
     Planetary surface gravity (at 1 bar) in cm/second^2.

  Returns
  -------
  T: temperature array

  Example:
  --------
  >>> import PT as pt
  >>> import scipy.constants as sc
  >>> import matplotlib.pyplot as plt
  >>> import numpy as np

  >>> Rsun = 6.995e8 # Sun radius in meters

  >>> # Pressure array (bars):
  >>> p = np.logspace(2, -5, 100)

  >>> # Physical (fixed for each planet) parameters:
  >>> Ts = 5040.0        # K
  >>> Ti =  100.0        # K
  >>> a  = 0.031 * sc.au # m
  >>> Rs = 0.756 * Rsun  # m
  >>> g  = 2192.8        # cm s-2

  >>> # Fitting parameters:
  >>> kappa  = -1.5   # log10(3e-2)
  >>> gamma1 = -0.8   # log10(0.158)
  >>> gamma2 = -0.8   # log10(0.158)
  >>> alpha  = 0.5
  >>> beta   = 1.0
  >>> params = [kappa, gamma1, gamma2, alpha, beta]
  >>> T0 = pt.PT(p, params, Rs, Ts, Ti, a, g)

  >>> plt.figure(1)
  >>> plt.clf()
  >>> plt.semilogy(T0, p, lw=2, color="b")
  >>> plt.ylim(p[0], p[-1])
  >>> plt.xlim(800, 2000)
  >>> plt.xlabel("Temperature  (K)")
  >>> plt.ylabel("Pressure  (bars)")

  Developers:
  -----------
  Madison Stemm      astromaddie@gmail.com
  Patricio Cubillos  pcubillos@fulbrightmail.org

  Modification History:
  ---------------------
  2014-09-12  Madison   Initial version, adapted from equations (13)-(16)
                        in Line et al. (2013), Apj, 775, 137.
  2014-12-10  patricio  Reviewed and updated code.
  2015-01-22  patricio  Receive log10 of free parameters now.
  '''

  # Unpack free parameters:
  kappa  = 10**(params[0])
  gamma1 = 10**(params[1])
  gamma2 = 10**(params[2])
  alpha, beta = params[3], params[4]

  # Stellar input temperature (at top of atmosphere):
  T_irr = beta * (R_star / (2.0*sma))**0.5 * T_star

  # Gray IR optical depth:
  tau = kappa * (pressure*1e6) / grav # Convert bars to barye (CGS)

  xi1 = xi(gamma1, tau)
  xi2 = xi(gamma2, tau)

  # Temperature profile (Eq. 13 of Line et al. 2013):
  temperature = (0.75 * (T_int**4 * (2.0/3.0 + tau) +
                         T_irr**4 * (1-alpha) * xi1 +
                         T_irr**4 * alpha     * xi2 ) )**0.25

  return temperature

def xi(gamma, tau):
  """
  Calculate Equation (14) of Line et al. (2013) Apj 775, 137

  Parameters:
  -----------
  gamma: Float
     Visible-to-thermal stream Planck mean opacity ratio.
  tau: 1D float ndarray
     Gray IR optical depth.

  Modification History:
  ---------------------
  2014-12-10  patricio  Initial implemetation.
  """
  return (2.0/3) * (1 + (1/gamma) * (1 + (0.5*gamma*tau-1)*np.exp(-gamma*tau)) +
                    gamma*(1 - 0.5*tau**2) * sp.expn(2, gamma*tau)             )

def genPT(log10kappa, log10gamma1, log10gamma2, alpha, beta, pmin, pmax, Rstar,
          Tstar, sma, surfgrav, Tint=0, nlayers=50):
    """
    This function generatures a pressure-temperature (PT) profile for given
    input parameters using the model of Line et al. (2013).

    Note that this uses PT.py and reader.py from BART, the Bayesian Atmospheric
    Radiative Transfer code, which has an open-source, reproducible-research license.


    Inputs
    ------
    log10kappa:   Planck thermal IR opacity (in units cm^2/gr)
    log10gamma1:  Visible-to-thermal stream Planck mean opacity ratio.
    log10gamma2:  Visible-to-thermal stream Planck mean opacity ratio.
    alpha:         Visible-stream partition (0.0--1.0).
    beta:          A 'catch-all' for albedo, emissivity, and day-night
                      redistribution (on the order of unity).
    pmin:          Minimum pressure at the top of the atmosphere (in bars).
    pmax:          Pressure at the surface of the planet (in bars).
    Rstar:         Radius of the host star (in solar radii).
    Tstar:         Temperature of the host star (in Kelvin).
    sma:           Semimajor axis of planet (in AU).
    surfgrav:      Planetary gravity at 1 bar of pressure (in cm/s^2).
    Tint:          Planetary internal heat flux (in Kelvin).
                   Default is 0 since these planets will have negligible flux
                   compared to host star's incident flux.
    nlayers:       Number of layers in the atmosphere.

    Returns
    -------
    PTprof:        2D array containing the pressure (equally spaced in
                   logspace) and temperature at each pressure.
                   PTprof[i] gives the i-th layer's [pressure, temperature]
                   PTprof[:,0] gives the pressure at all layers.
                   PTprof[:,1] gives the temperature at all layers.
    """
    # Put PT parameters into array
    params   = np.array([log10kappa, log10gamma1, log10gamma2, alpha, beta])
    # Range of pressures equally spaced in logspace
    pressure = np.logspace(np.log10(pmax), np.log10(pmin), nlayers)

    # Generate temperature profile
    #                  [bars]   [params]        [meters]            [K]
    #                  [K]      [meters]        [cm/s^2]
    temp = PT_line(pressure, params, Rstar * const.R_sun.value, Tstar,
                      Tint, sma * const.au.value, surfgrav)

    # Put temperature and pressure into array, return
    PTprof      = np.zeros((nlayers, 2), dtype=float)
    PTprof[:,0] = pressure
    PTprof[:,1] = temp

    return PTprof

def makeclouds(abun, PTprof, nT=400):
    """
    This function computes the amount of condensation that will occur for a species.
    Note that clouds are only computed within the range of our data
    Allowed species: H2O (255.9 K <= T <= 573 K), NH3 (164 K <= T <= 371.5 K)
    Sources:
    H2O: https://webbook.nist.gov/cgi/cbook.cgi?ID=C7732185&Units=SI&Mask=4#Thermo-Phase
    NH3: https://webbook.nist.gov/cgi/cbook.cgi?ID=C7664417&Units=SI&Mask=4#Thermo-Phase

    Inputs
    ------
    abun    - Abundance of H2O and NH3.
    PTprof  - PT profile as created by genPT().
    nT      - Int. Number of points to calculate SVP.
    
    Outputs
    -------
    clouds  - Abundance of H2O and NH3 clouds at each layer in the atmosphere.
    """
    # Repeat abundance array to match PTprof shape
    #abunarr = np.repeat(abun, len(PTprof[:,0]))

    # Arrays of Antoine equation parameters for Saturation Vapor Pressure (SVP)
    #                Tlo,   Thi,  A,       B,          C
    h2o = np.array([[255.9, 373. , 4.6543 , 1435.264,  -64.848], 
                    [273. , 303. , 5.40221, 1838.675,  -31.737], 
                    [304. , 333. , 5.20389, 1733.926,  -39.485], 
                    [334. , 363. , 5.0768 , 1659.793,  -45.854], 
                    [344. , 373. , 5.08354, 1663.125,  -45.622], 
                    [379. , 573. , 3.55959,  643.748, -198.043]])
    nh3 = np.array([[164. , 239.6, 3.18757,  506.713,  -80.78 ], 
                    [239.6, 371.5, 4.86886, 1113.928,  -10.409]])

    Th2o = np.linspace(np.amin(h2o[:,0]), np.amax(h2o[:,1]), num=nT)
    Tnh3 = np.linspace(np.amin(nh3[:,0]), np.amax(h2o[:,1]), num=nT)

    # Use Antoine equation to calculate SVP
    # p[bar] = 10^(A - (B / (T[K] + C)))
    svph2o = np.zeros(nT)
    svpnh3 = np.zeros(nT)

    # Different temperature regimes for H2O
    ir1 =  Th2o < h2o[1,0]
    ir2 = (Th2o >= h2o[1,0])              * (Th2o <= (h2o[2,0]+h2o[1,1])/2.)
    ir3 = (Th2o > (h2o[2,0]+h2o[1,1])/2.) * (Th2o <= (h2o[3,0]+h2o[2,1])/2.)
    ir4 = (Th2o > (h2o[3,0]+h2o[2,1])/2.) * (Th2o <   h2o[4,0])
    ir5 = (Th2o >= h2o[4,0]) * (Th2o<=h2o[3,1])
    ir6 = (Th2o >  h2o[3,1]) * (Th2o<=h2o[4,1])
    ir7 =  Th2o >= h2o[5,0]
    # Fill in SVP for H2O
    svph2o[ir1] =  10**(h2o[0,2] - (h2o[0,3] / (Th2o[ir1] + h2o[0,4])))
    svph2o[ir2] =  10**(h2o[1,2] - (h2o[1,3] / (Th2o[ir2] + h2o[1,4])))
    svph2o[ir3] =  10**(h2o[2,2] - (h2o[2,3] / (Th2o[ir3] + h2o[2,4])))
    svph2o[ir4] =  10**(h2o[3,2] - (h2o[3,3] / (Th2o[ir4] + h2o[3,4])))
    svph2o[ir5] = (10**(h2o[3,2] - (h2o[3,3] / (Th2o[ir5] + h2o[3,4]))) + \
                   10**(h2o[4,2] - (h2o[4,3] / (Th2o[ir5] + h2o[4,4])))) / 2.
    svph2o[ir6] =  10**(h2o[4,2] - (h2o[4,3] / (Th2o[ir6] + h2o[4,4])))
    svph2o[ir7] =  10**(h2o[5,2] - (h2o[5,3] / (Th2o[ir7] + h2o[5,4])))

    # Different temperature regimes for NH3
    ir1 = Tnh3 <= nh3[0,1]
    ir2 = Tnh3 >  nh3[1,0]
    # Fill in SVP for NH3
    svpnh3[ir1] = 10**(nh3[0,2] - (nh3[0,3] / (Tnh3[ir1] + nh3[0,4])))
    svpnh3[ir2] = 10**(nh3[1,2] - (nh3[1,3] / (Tnh3[ir2] + nh3[1,4])))

    # Remove the 0 values as they will mess up interpolation
    Th2o   =   Th2o[svph2o!=0]
    Tnh3   =   Tnh3[svpnh3!=0]
    svph2o = svph2o[svph2o!=0]
    svpnh3 = svpnh3[svpnh3!=0]

    # Functions and indices for interpolation
    # We only allow interpolation where the data is defined
    h2osvpinterp = si.interp1d(Th2o, svph2o)
    ih2o         = (PTprof[:,1] >= Th2o[0]) * (PTprof[:,1] <= Th2o[-1])
    nh3svpinterp = si.interp1d(Tnh3, svpnh3)
    inh3         = (PTprof[:,1] >= Tnh3[0]) * (PTprof[:,1] <= Tnh3[-1])
    # Interpolate to the desired values
    laysvp          = np.zeros((len(PTprof[:,1]), 2))
    laysvp[ih2o, 0] = h2osvpinterp(PTprof[:,1][ih2o])
    laysvp[inh3, 1] = nh3svpinterp(PTprof[:,1][inh3])

    # Arrays to hold cloud info
    clouds    = np.zeros((len(PTprof[:,0]), 2))
    cloudsh2o = np.zeros( len(PTprof[:,0]))
    cloudsnh3 = np.zeros( len(PTprof[:,0]))
    # Indices to determine where SVP is defined
    ih2o = laysvp[:,0]!=0
    inh3 = laysvp[:,1]!=0

    # Calculate cloud abundances
    cloudsh2o[ih2o] = abun[0] * ((abun[0] * PTprof[:,0])[ih2o] - laysvp[ih2o,0]) / \
                                 (abun[0] * PTprof[:,0])[ih2o]
    cloudsnh3[inh3] = abun[1] * ((abun[1] * PTprof[:,0])[inh3] - laysvp[inh3,1]) / \
                                 (abun[1] * PTprof[:,0])[inh3]

    # Package array, remove negatives
    clouds[:,0] = cloudsh2o
    clouds[:,1] = cloudsnh3
    clouds[clouds<0] = 0

    return clouds

In [2]:
def setobsparams(instr):
    """
    This function defines the observational parameters for various observing
    modes of various telescopes.

    Work in progress. LUVOIR_ECLIPS and LUVOIR_ECSPEC are based on design parameters 
    for LUVOIR, but at a much higher resolution. Note that it is split into multiple 
    channels to allow for scattering calculations.

    Some of the output descriptions are adapted/taken from 
        https://psg.gsfc.nasa.gov/helpapi.php
    See that page for more details. Note there is not a direct match in parameter names, 
    but they are close.


    Inputs
    ------
    instr   - String. Specifies instrument to observe with.

    Outputs
    -------
    rangelo     - Lower range of wavelength [um]
    rangehi     - Upper arnge of wavelength [um]
    resolution  - Resolving power of instrument
    trans       - PSG specification of altitude and atmosphere water %
    transshow   - Y/N flag to determine whether to simulate telluric 
                  absorption of Earth's atmosphere
    transapply  - Y/N flag to determine whether to return the observation as 
                  observed w/ telluric absorption, or to divide by telluric transmittance
    radunits    - Radiation unit of produced spectrum according to PSG's specifications.
    telescope   - Type of telescope. 'SINGLE', 'CORONA', 'ARRAY'
    beam        - FWHM of instrument's beam.
    beamunit    - Unit of `beam`. 'arcsec', 'arcmin', 'degree', 'km', 
                  'diameter' (size in terms of planet diameter), 
                  'diffrac' (defined by telescope diameter and center wavelength)
    diameter    - Diameter of telescope. [m]
    telescope1  - For interferometers, # of telescopes. For coronagraphs, instrument contrast.
    telescope2  - For coronagraphic observations, exozodi level
    telescope3  - For coronagraphic observations, innerworking angle in units [L/D]
    noise       - Noise model specifier. 'NO' (none), 'TRX' (receiver temp/radio), 
                  'RMS' (constant noise in rad units), 'BKG' (constant noise plus background), 
                  'NEP' (power equivalent to noise detector model), 
                  'D' (detectability noise detector model), 'CCD' (image sensor)
    noise1      - First noise model parameter. 
                  For RMS, 1-sigma noise.
                  For TRX, the receiver temperature
                  For BKG, the 1-sigma noise
                  For NEP, the sensitivity in W/sqrt(Hz)
                  For DET, the sensitivity in cm.sqrt(Hz)/W
                  For CCD, the read noise [e-]
    noise2      - Second noise model parameter
                  For RMS, not used
                  For TRX, the sideband g-factor
                  For BKG, the not used
                  For NEP, not used
                  For DET, the pixel size [um]
                  For CCD, the dark rate [e-/s]
    noiseotemp  - Temperature of telescpe+instrument optics [K]
    noiseoeff   - Total throughput of telescope+instrument
    noiseoemis  - Emissivity of telescope+optics
    noisetime   - Exposure time per frame [sec]
    noiseframe  - Number of exposures
    noisepixel  - Number of pixels that encompass the beam
    """
    # JWST NIRISS SOSS, NIRSpec G395M, MIRI LRS:
    # https://arxiv.org/pdf/1611.08608.pdf
    # JWST observation info https://arxiv.org/pdf/1411.1754.pdf
    # https://arxiv.org/pdf/1803.04985.pdf
    #
    """if instr=='JWST-MIRI-LRS':
        rangelo    =
        rangehi    =
        resolution =
        trans      =
        transshow  =
        transapply =
        radunits   =
        telescope  =
        beam       =
        beamunit   =
        diameter   =
        telescope1 =
        telescope2 =
        telescope3 =
        noise      =
        noisetime  =
        noiseframe =
        noisepixel =
        noise1     =
        noise2     =
        noiseoeff  =
        noiseoemis =
        noiseotemp =
        instrument =
    elif instr=='JWST-MIRI-MRS':
        rangelo    =
        rangehi    =
        resolution =
        trans      =
        transshow  =
        transapply =
        radunits   =
        telescope  =
        beam       =
        beamunit   =
        diameter   =
        telescope1 =
        telescope2 =
        telescope3 =
        noise      =
        noisetime  =
        noiseframe =
        noisepixel =
        noise1     =
        noise2     =
        noiseoeff  =
        noiseoemis =
        noiseotemp =
        instrument =
    elif instr=='JWST-NIRISS-SOSS':
        rangelo    =
        rangehi    =
        resolution =
        trans      =
        transshow  =
        transapply =
        radunits   =
        telescope  =
        beam       =
        beamunit   =
        diameter   =
        telescope1 =
        telescope2 =
        telescope3 =
        noise      =
        noisetime  =
        noiseframe =
        noisepixel =
        noise1     =
        noise2     =
        noiseoeff  =
        noiseoemis =
        noiseotemp =
        instrument =
    elif instr=='JWST-NIRSpec-G235H':
        rangelo    =
        rangehi    =
        resolution =
        trans      =
        transshow  =
        transapply =
        radunits   =
        telescope  =
        beam       =
        beamunit   =
        diameter   =
        telescope1 =
        telescope2 =
        telescope3 =
        noise      =
        noisetime  =
        noiseframe =
        noisepixel =
        noise1     =
        noise2     =
        noiseoeff  =
        noiseoemis =
        noiseotemp =
        instrument =
    elif instr=='JWST-NIRSpec-G395H':
        rangelo    =
        rangehi    =
        resolution =
        trans      =
        transshow  =
        transapply =
        radunits   =
        telescope  =
        beam       =
        beamunit   =
        diameter   =
        telescope1 =
        telescope2 =
        telescope3 =
        noise      =
        noisetime  =
        noiseframe =
        noisepixel =
        noise1     =
        noise2     =
        noiseoeff  =
        noiseoemis =
        noiseotemp =
        instrument =
    elif instr=='JWST-NIRCam-F322W2':
        rangelo    =
        rangehi    =
        resolution =
        trans      =
        transshow  =
        transapply =
        radunits   =
        telescope  =
        beam       =
        beamunit   =
        diameter   =
        telescope1 =
        telescope2 =
        telescope3 =
        noise      =
        noisetime  =
        noiseframe =
        noisepixel =
        noise1     =
        noise2     =
        noiseoeff  =
        noiseoemis =
        noiseotemp =
        instrument =
    elif instr=='JWST-NIRCam-F444W':
        rangelo    =
        rangehi    =
        resolution =
        trans      =
        transshow  =
        transapply =
        radunits   =
        telescope  =
        beam       =
        beamunit   =
        diameter   =
        telescope1 =
        telescope2 =
        telescope3 =
        noise      =
        noisetime  =
        noiseframe =
        noisepixel =
        noise1     =
        noise2     =
        noiseoeff  =
        noiseoemis =
        noiseotemp =
        instrument =
    """
    if instr	==	'LUVOIR_ECLIPS_02_03':
        rangelo	    =	0.2
        rangehi     =   0.3
        resolution	=	1900
        telescope	=	'CORONA'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-10
        telescope2	=	3
        telescope3	=	2
        noise	    =	'CCD'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.007@2.00e-1,0.008@2.15e-1,0.010@2.40e-1,0.013@2.70e-1,' + \
                        '0.016@3.05e-1,0.018@3.45e-1,0.020@3.80e-1,0.043@4.30e-1,' + \
                        '0.049@4.85e-1,0.049@5.50e-1,0.044@6.25e-1,0.036@7.15e-1,' + \
                        '0.026@8.05e-1,0.044@9.20e-1,0.069@1.06e+0,0.080@1.22e+0,' + \
                        '0.086@1.41e+0,0.090@1.62e+0,0.092@1.87e+0,0.094@2.16e+0,' + \
                        '0.095@2.40e+0,0.096@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	120
        noiseframe	=	240
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'erg'
    elif instr	==	'LUVOIR_ECLIPS_03_04':
        rangelo	    =	0.3
        rangehi     =   0.4
        resolution	=	1900
        telescope	=	'CORONA'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-10
        telescope2	=	3
        telescope3	=	2
        noise	    =	'CCD'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.007@2.00e-1,0.008@2.15e-1,0.010@2.40e-1,0.013@2.70e-1,' + \
                        '0.016@3.05e-1,0.018@3.45e-1,0.020@3.80e-1,0.043@4.30e-1,' + \
                        '0.049@4.85e-1,0.049@5.50e-1,0.044@6.25e-1,0.036@7.15e-1,' + \
                        '0.026@8.05e-1,0.044@9.20e-1,0.069@1.06e+0,0.080@1.22e+0,' + \
                        '0.086@1.41e+0,0.090@1.62e+0,0.092@1.87e+0,0.094@2.16e+0,' + \
                        '0.095@2.40e+0,0.096@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	120
        noiseframe	=	240
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'erg'
    elif instr	==	'LUVOIR_ECLIPS_04_06':
        rangelo	    =	0.4
        rangehi     =   0.6
        resolution	=	1900
        telescope	=	'CORONA'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-10
        telescope2	=	3
        telescope3	=	2
        noise	    =	'CCD'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.007@2.00e-1,0.008@2.15e-1,0.010@2.40e-1,0.013@2.70e-1,' + \
                        '0.016@3.05e-1,0.018@3.45e-1,0.020@3.80e-1,0.043@4.30e-1,' + \
                        '0.049@4.85e-1,0.049@5.50e-1,0.044@6.25e-1,0.036@7.15e-1,' + \
                        '0.026@8.05e-1,0.044@9.20e-1,0.069@1.06e+0,0.080@1.22e+0,' + \
                        '0.086@1.41e+0,0.090@1.62e+0,0.092@1.87e+0,0.094@2.16e+0,' + \
                        '0.095@2.40e+0,0.096@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	120
        noiseframe	=	240
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'erg'
    elif instr	==	'LUVOIR_ECLIPS_06_08':
        rangelo	    =	0.6
        rangehi     =   0.8
        resolution	=	1900
        telescope	=	'CORONA'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-10
        telescope2	=	3
        telescope3	=	2
        noise	    =	'CCD'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.007@2.00e-1,0.008@2.15e-1,0.010@2.40e-1,0.013@2.70e-1,' + \
                        '0.016@3.05e-1,0.018@3.45e-1,0.020@3.80e-1,0.043@4.30e-1,' + \
                        '0.049@4.85e-1,0.049@5.50e-1,0.044@6.25e-1,0.036@7.15e-1,' + \
                        '0.026@8.05e-1,0.044@9.20e-1,0.069@1.06e+0,0.080@1.22e+0,' + \
                        '0.086@1.41e+0,0.090@1.62e+0,0.092@1.87e+0,0.094@2.16e+0,' + \
                        '0.095@2.40e+0,0.096@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	120
        noiseframe	=	240
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'erg'
    elif instr	==	'LUVOIR_ECLIPS_08_13':
        rangelo	    =	0.8
        rangehi     =   1.3
        resolution	=	1900
        telescope	=	'CORONA'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-10
        telescope2	=	3
        telescope3	=	2
        noise	    =	'CCD'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.007@2.00e-1,0.008@2.15e-1,0.010@2.40e-1,0.013@2.70e-1,' + \
                        '0.016@3.05e-1,0.018@3.45e-1,0.020@3.80e-1,0.043@4.30e-1,' + \
                        '0.049@4.85e-1,0.049@5.50e-1,0.044@6.25e-1,0.036@7.15e-1,' + \
                        '0.026@8.05e-1,0.044@9.20e-1,0.069@1.06e+0,0.080@1.22e+0,' + \
                        '0.086@1.41e+0,0.090@1.62e+0,0.092@1.87e+0,0.094@2.16e+0,' + \
                        '0.095@2.40e+0,0.096@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	120
        noiseframe	=	240
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'erg'
    elif instr	==	'LUVOIR_ECLIPS_13_2':
        rangelo	    =	1.3
        rangehi     =   2.0
        resolution	=	1900
        telescope	=	'CORONA'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-10
        telescope2	=	3
        telescope3	=	2
        noise	    =	'CCD'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.007@2.00e-1,0.008@2.15e-1,0.010@2.40e-1,0.013@2.70e-1,' + \
                        '0.016@3.05e-1,0.018@3.45e-1,0.020@3.80e-1,0.043@4.30e-1,' + \
                        '0.049@4.85e-1,0.049@5.50e-1,0.044@6.25e-1,0.036@7.15e-1,' + \
                        '0.026@8.05e-1,0.044@9.20e-1,0.069@1.06e+0,0.080@1.22e+0,' + \
                        '0.086@1.41e+0,0.090@1.62e+0,0.092@1.87e+0,0.094@2.16e+0,' + \
                        '0.095@2.40e+0,0.096@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	120
        noiseframe	=	240
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'erg'
    elif instr	==	'LUVOIR_ECLIPS_2_4':
        rangelo	    =	2.0
        rangehi     =   4.0
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-10
        telescope2	=	3
        telescope3	=	2e-6
        noise	    =	'NO'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.007@2.00e-1,0.008@2.15e-1,0.010@2.40e-1,0.013@2.70e-1,' + \
                        '0.016@3.05e-1,0.018@3.45e-1,0.020@3.80e-1,0.043@4.30e-1,' + \
                        '0.049@4.85e-1,0.049@5.50e-1,0.044@6.25e-1,0.036@7.15e-1,' + \
                        '0.026@8.05e-1,0.044@9.20e-1,0.069@1.06e+0,0.080@1.22e+0,' + \
                        '0.086@1.41e+0,0.090@1.62e+0,0.092@1.87e+0,0.094@2.16e+0,' + \
                        '0.095@2.40e+0,0.096@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	120
        noiseframe	=	240
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'erg'
    elif instr	==	'LUVOIR_ECLIPS_4_8':
        rangelo	    =	4.0
        rangehi     =   8.0
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-10
        telescope2	=	3
        telescope3	=	2e-6
        noise	    =	'NO'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.007@2.00e-1,0.008@2.15e-1,0.010@2.40e-1,0.013@2.70e-1,' + \
                        '0.016@3.05e-1,0.018@3.45e-1,0.020@3.80e-1,0.043@4.30e-1,' + \
                        '0.049@4.85e-1,0.049@5.50e-1,0.044@6.25e-1,0.036@7.15e-1,' + \
                        '0.026@8.05e-1,0.044@9.20e-1,0.069@1.06e+0,0.080@1.22e+0,' + \
                        '0.086@1.41e+0,0.090@1.62e+0,0.092@1.87e+0,0.094@2.16e+0,' + \
                        '0.095@2.40e+0,0.096@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	120
        noiseframe	=	240
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'erg'
    elif instr	==	'LUVOIR_ECLIPS_8_12':
        rangelo	    =	8.0
        rangehi     =   12.0
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-10
        telescope2	=	3
        telescope3	=	2e-6
        noise	    =	'NO'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.007@2.00e-1,0.008@2.15e-1,0.010@2.40e-1,0.013@2.70e-1,' + \
                        '0.016@3.05e-1,0.018@3.45e-1,0.020@3.80e-1,0.043@4.30e-1,' + \
                        '0.049@4.85e-1,0.049@5.50e-1,0.044@6.25e-1,0.036@7.15e-1,' + \
                        '0.026@8.05e-1,0.044@9.20e-1,0.069@1.06e+0,0.080@1.22e+0,' + \
                        '0.086@1.41e+0,0.090@1.62e+0,0.092@1.87e+0,0.094@2.16e+0,' + \
                        '0.095@2.40e+0,0.096@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	120
        noiseframe	=	240
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'erg'
    elif instr	==	'LUVOIR_ECLIPS_12_20':
        rangelo	    =	12.0
        rangehi     =   20.0
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-10
        telescope2	=	3
        telescope3	=	2e-6
        noise	    =	'NO'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.007@2.00e-1,0.008@2.15e-1,0.010@2.40e-1,0.013@2.70e-1,' + \
                        '0.016@3.05e-1,0.018@3.45e-1,0.020@3.80e-1,0.043@4.30e-1,' + \
                        '0.049@4.85e-1,0.049@5.50e-1,0.044@6.25e-1,0.036@7.15e-1,' + \
                        '0.026@8.05e-1,0.044@9.20e-1,0.069@1.06e+0,0.080@1.22e+0,' + \
                        '0.086@1.41e+0,0.090@1.62e+0,0.092@1.87e+0,0.094@2.16e+0,' + \
                        '0.095@2.40e+0,0.096@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	120
        noiseframe	=	240
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'erg'
    elif instr	==	'LUVOIR_ECLIPS_20_40':
        rangelo	    =	20.0
        rangehi     =   40.0
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-10
        telescope2	=	3
        telescope3	=	2
        noise	    =	'NO'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.007@2.00e-1,0.008@2.15e-1,0.010@2.40e-1,0.013@2.70e-1,' + \
                        '0.016@3.05e-1,0.018@3.45e-1,0.020@3.80e-1,0.043@4.30e-1,' + \
                        '0.049@4.85e-1,0.049@5.50e-1,0.044@6.25e-1,0.036@7.15e-1,' + \
                        '0.026@8.05e-1,0.044@9.20e-1,0.069@1.06e+0,0.080@1.22e+0,' + \
                        '0.086@1.41e+0,0.090@1.62e+0,0.092@1.87e+0,0.094@2.16e+0,' + \
                        '0.095@2.40e+0,0.096@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	120
        noiseframe	=	240
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'erg'
    elif instr	==	'LUVOIR_ECLIPS_40_80':
        rangelo	    =	40.0
        rangehi     =   80.0
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-10
        telescope2	=	3
        telescope3	=	2
        noise	    =	'NO'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.007@2.00e-1,0.008@2.15e-1,0.010@2.40e-1,0.013@2.70e-1,' + \
                        '0.016@3.05e-1,0.018@3.45e-1,0.020@3.80e-1,0.043@4.30e-1,' + \
                        '0.049@4.85e-1,0.049@5.50e-1,0.044@6.25e-1,0.036@7.15e-1,' + \
                        '0.026@8.05e-1,0.044@9.20e-1,0.069@1.06e+0,0.080@1.22e+0,' + \
                        '0.086@1.41e+0,0.090@1.62e+0,0.092@1.87e+0,0.094@2.16e+0,' + \
                        '0.095@2.40e+0,0.096@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	120
        noiseframe	=	240
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'erg'
    elif instr	==	'LUVOIR_ECLIPS_80_160':
        rangelo	    =	80.0
        rangehi     =   160.0
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-10
        telescope2	=	3
        telescope3	=	2
        noise	    =	'NO'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.007@2.00e-1,0.008@2.15e-1,0.010@2.40e-1,0.013@2.70e-1,' + \
                        '0.016@3.05e-1,0.018@3.45e-1,0.020@3.80e-1,0.043@4.30e-1,' + \
                        '0.049@4.85e-1,0.049@5.50e-1,0.044@6.25e-1,0.036@7.15e-1,' + \
                        '0.026@8.05e-1,0.044@9.20e-1,0.069@1.06e+0,0.080@1.22e+0,' + \
                        '0.086@1.41e+0,0.090@1.62e+0,0.092@1.87e+0,0.094@2.16e+0,' + \
                        '0.095@2.40e+0,0.096@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	120
        noiseframe	=	240
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'erg'
    elif instr	==	'LUVOIR_ECLIPS_160_320':
        rangelo	    =	160.0
        rangehi     =   320.0
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-10
        telescope2	=	3
        telescope3	=	2
        noise	    =	'NO'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.007@2.00e-1,0.008@2.15e-1,0.010@2.40e-1,0.013@2.70e-1,' + \
                        '0.016@3.05e-1,0.018@3.45e-1,0.020@3.80e-1,0.043@4.30e-1,' + \
                        '0.049@4.85e-1,0.049@5.50e-1,0.044@6.25e-1,0.036@7.15e-1,' + \
                        '0.026@8.05e-1,0.044@9.20e-1,0.069@1.06e+0,0.080@1.22e+0,' + \
                        '0.086@1.41e+0,0.090@1.62e+0,0.092@1.87e+0,0.094@2.16e+0,' + \
                        '0.095@2.40e+0,0.096@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	120
        noiseframe	=	240
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'erg'
    elif instr	==	'LUVOIR_ECLIPS_320_640':
        rangelo	    =	320.0
        rangehi     =   640.0
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-10
        telescope2	=	3
        telescope3	=	2
        noise	    =	'NO'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.007@2.00e-1,0.008@2.15e-1,0.010@2.40e-1,0.013@2.70e-1,' + \
                        '0.016@3.05e-1,0.018@3.45e-1,0.020@3.80e-1,0.043@4.30e-1,' + \
                        '0.049@4.85e-1,0.049@5.50e-1,0.044@6.25e-1,0.036@7.15e-1,' + \
                        '0.026@8.05e-1,0.044@9.20e-1,0.069@1.06e+0,0.080@1.22e+0,' + \
                        '0.086@1.41e+0,0.090@1.62e+0,0.092@1.87e+0,0.094@2.16e+0,' + \
                        '0.095@2.40e+0,0.096@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	120
        noiseframe	=	240
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'erg'
    elif instr	==	'LUVOIR_ECSPEC_02_03':
        rangelo	    =	0.2
        rangehi     =   0.3
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-09
        telescope2	=	3
        telescope3	=	2
        noise	    =	'CCD'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.050@2.00e-1,0.053@2.15e-1,0.068@2.40e-1,0.085@2.70e-1,' + \
                        '0.103@3.05e-1,0.122@3.45e-1,0.132@3.80e-1,0.290@4.30e-1,' + \
                        '0.324@4.85e-1,0.324@5.50e-1,0.296@6.25e-1,0.239@7.15e-1,' + \
                        '0.171@8.05e-1,0.294@9.20e-1,0.459@1.06e+0,0.534@1.22e+0,' + \
                        '0.573@1.41e+0,0.598@1.62e+0,0.615@1.87e+0,'               + \
                        '0.628@2.16e+0,0.636@2.40e+0,0.640@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	0.2
        noiseframe	=	1.44e5
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'rel'
    elif instr	==	'LUVOIR_ECSPEC_03_04':
        rangelo	    =	0.3
        rangehi     =   0.4
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-09
        telescope2	=	3
        telescope3	=	2
        noise	    =	'CCD'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.050@2.00e-1,0.053@2.15e-1,0.068@2.40e-1,0.085@2.70e-1,' + \
                        '0.103@3.05e-1,0.122@3.45e-1,0.132@3.80e-1,0.290@4.30e-1,' + \
                        '0.324@4.85e-1,0.324@5.50e-1,0.296@6.25e-1,0.239@7.15e-1,' + \
                        '0.171@8.05e-1,0.294@9.20e-1,0.459@1.06e+0,0.534@1.22e+0,' + \
                        '0.573@1.41e+0,0.598@1.62e+0,0.615@1.87e+0,'               + \
                        '0.628@2.16e+0,0.636@2.40e+0,0.640@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	0.2
        noiseframe	=	1.44e5
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'rel'
    elif instr	==	'LUVOIR_ECSPEC_04_06':
        rangelo	    =	0.4
        rangehi     =   0.6
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-09
        telescope2	=	3
        telescope3	=	2
        noise	    =	'CCD'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.050@2.00e-1,0.053@2.15e-1,0.068@2.40e-1,0.085@2.70e-1,' + \
                        '0.103@3.05e-1,0.122@3.45e-1,0.132@3.80e-1,0.290@4.30e-1,' + \
                        '0.324@4.85e-1,0.324@5.50e-1,0.296@6.25e-1,0.239@7.15e-1,' + \
                        '0.171@8.05e-1,0.294@9.20e-1,0.459@1.06e+0,0.534@1.22e+0,' + \
                        '0.573@1.41e+0,0.598@1.62e+0,0.615@1.87e+0,'               + \
                        '0.628@2.16e+0,0.636@2.40e+0,0.640@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	0.2
        noiseframe	=	1.44e5
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'rel'
    elif instr	==	'LUVOIR_ECSPEC_06_08':
        rangelo	    =	0.6
        rangehi     =   0.8
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-09
        telescope2	=	3
        telescope3	=	2
        noise	    =	'CCD'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.050@2.00e-1,0.053@2.15e-1,0.068@2.40e-1,0.085@2.70e-1,' + \
                        '0.103@3.05e-1,0.122@3.45e-1,0.132@3.80e-1,0.290@4.30e-1,' + \
                        '0.324@4.85e-1,0.324@5.50e-1,0.296@6.25e-1,0.239@7.15e-1,' + \
                        '0.171@8.05e-1,0.294@9.20e-1,0.459@1.06e+0,0.534@1.22e+0,' + \
                        '0.573@1.41e+0,0.598@1.62e+0,0.615@1.87e+0,'               + \
                        '0.628@2.16e+0,0.636@2.40e+0,0.640@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	0.2
        noiseframe	=	1.44e5
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'rel'
    elif instr	==	'LUVOIR_ECSPEC_08_13':
        rangelo	    =	0.8
        rangehi     =   1.3
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-09
        telescope2	=	3
        telescope3	=	2
        noise	    =	'CCD'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.050@2.00e-1,0.053@2.15e-1,0.068@2.40e-1,0.085@2.70e-1,' + \
                        '0.103@3.05e-1,0.122@3.45e-1,0.132@3.80e-1,0.290@4.30e-1,' + \
                        '0.324@4.85e-1,0.324@5.50e-1,0.296@6.25e-1,0.239@7.15e-1,' + \
                        '0.171@8.05e-1,0.294@9.20e-1,0.459@1.06e+0,0.534@1.22e+0,' + \
                        '0.573@1.41e+0,0.598@1.62e+0,0.615@1.87e+0,'               + \
                        '0.628@2.16e+0,0.636@2.40e+0,0.640@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	0.2
        noiseframe	=	1.44e5
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'rel'
    elif instr	==	'LUVOIR_ECSPEC_13_2':
        rangelo	    =	1.3
        rangehi     =   2.0
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-09
        telescope2	=	3
        telescope3	=	2
        noise	    =	'CCD'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.050@2.00e-1,0.053@2.15e-1,0.068@2.40e-1,0.085@2.70e-1,' + \
                        '0.103@3.05e-1,0.122@3.45e-1,0.132@3.80e-1,0.290@4.30e-1,' + \
                        '0.324@4.85e-1,0.324@5.50e-1,0.296@6.25e-1,0.239@7.15e-1,' + \
                        '0.171@8.05e-1,0.294@9.20e-1,0.459@1.06e+0,0.534@1.22e+0,' + \
                        '0.573@1.41e+0,0.598@1.62e+0,0.615@1.87e+0,'               + \
                        '0.628@2.16e+0,0.636@2.40e+0,0.640@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	0.2
        noiseframe	=	1.44e5
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'rel'
    elif instr	==	'LUVOIR_ECSPEC_2_4':
        rangelo	    =	2.0
        rangehi     =   4.0
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-09
        telescope2	=	3
        telescope3	=	2
        noise	    =	'NO'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.050@2.00e-1,0.053@2.15e-1,0.068@2.40e-1,0.085@2.70e-1,' + \
                        '0.103@3.05e-1,0.122@3.45e-1,0.132@3.80e-1,0.290@4.30e-1,' + \
                        '0.324@4.85e-1,0.324@5.50e-1,0.296@6.25e-1,0.239@7.15e-1,' + \
                        '0.171@8.05e-1,0.294@9.20e-1,0.459@1.06e+0,0.534@1.22e+0,' + \
                        '0.573@1.41e+0,0.598@1.62e+0,0.615@1.87e+0,'               + \
                        '0.628@2.16e+0,0.636@2.40e+0,0.640@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	0.2
        noiseframe	=	1.44e5
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'rel'
    elif instr	==	'LUVOIR_ECSPEC_4_8':
        rangelo	    =	4.0
        rangehi     =   8.0
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-09
        telescope2	=	3
        telescope3	=	2
        noise	    =	'NO'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.050@2.00e-1,0.053@2.15e-1,0.068@2.40e-1,0.085@2.70e-1,' + \
                        '0.103@3.05e-1,0.122@3.45e-1,0.132@3.80e-1,0.290@4.30e-1,' + \
                        '0.324@4.85e-1,0.324@5.50e-1,0.296@6.25e-1,0.239@7.15e-1,' + \
                        '0.171@8.05e-1,0.294@9.20e-1,0.459@1.06e+0,0.534@1.22e+0,' + \
                        '0.573@1.41e+0,0.598@1.62e+0,0.615@1.87e+0,'               + \
                        '0.628@2.16e+0,0.636@2.40e+0,0.640@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	0.2
        noiseframe	=	1.44e5
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'rel'
    elif instr	==	'LUVOIR_ECSPEC_8_12':
        rangelo	    =	8.0
        rangehi     =   12.0
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-09
        telescope2	=	3
        telescope3	=	2
        noise	    =	'NO'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.050@2.00e-1,0.053@2.15e-1,0.068@2.40e-1,0.085@2.70e-1,' + \
                        '0.103@3.05e-1,0.122@3.45e-1,0.132@3.80e-1,0.290@4.30e-1,' + \
                        '0.324@4.85e-1,0.324@5.50e-1,0.296@6.25e-1,0.239@7.15e-1,' + \
                        '0.171@8.05e-1,0.294@9.20e-1,0.459@1.06e+0,0.534@1.22e+0,' + \
                        '0.573@1.41e+0,0.598@1.62e+0,0.615@1.87e+0,'               + \
                        '0.628@2.16e+0,0.636@2.40e+0,0.640@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	0.2
        noiseframe	=	1.44e5
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'rel'
    elif instr	==	'LUVOIR_ECSPEC_12_20':
        rangelo	    =	12.0
        rangehi     =   20.0
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-09
        telescope2	=	3
        telescope3	=	2
        noise	    =	'NO'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.050@2.00e-1,0.053@2.15e-1,0.068@2.40e-1,0.085@2.70e-1,' + \
                        '0.103@3.05e-1,0.122@3.45e-1,0.132@3.80e-1,0.290@4.30e-1,' + \
                        '0.324@4.85e-1,0.324@5.50e-1,0.296@6.25e-1,0.239@7.15e-1,' + \
                        '0.171@8.05e-1,0.294@9.20e-1,0.459@1.06e+0,0.534@1.22e+0,' + \
                        '0.573@1.41e+0,0.598@1.62e+0,0.615@1.87e+0,'               + \
                        '0.628@2.16e+0,0.636@2.40e+0,0.640@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	0.2
        noiseframe	=	1.44e5
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'rel'
    elif instr	==	'LUVOIR_ECSPEC_20_40':
        rangelo	    =	20.0
        rangehi     =   40.0
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-09
        telescope2	=	3
        telescope3	=	2
        noise	    =	'NO'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.050@2.00e-1,0.053@2.15e-1,0.068@2.40e-1,0.085@2.70e-1,' + \
                        '0.103@3.05e-1,0.122@3.45e-1,0.132@3.80e-1,0.290@4.30e-1,' + \
                        '0.324@4.85e-1,0.324@5.50e-1,0.296@6.25e-1,0.239@7.15e-1,' + \
                        '0.171@8.05e-1,0.294@9.20e-1,0.459@1.06e+0,0.534@1.22e+0,' + \
                        '0.573@1.41e+0,0.598@1.62e+0,0.615@1.87e+0,'               + \
                        '0.628@2.16e+0,0.636@2.40e+0,0.640@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	0.2
        noiseframe	=	1.44e5
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'rel'
    elif instr	==	'LUVOIR_ECSPEC_40_80':
        rangelo	    =	40.0
        rangehi     =   80.0
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-09
        telescope2	=	3
        telescope3	=	2
        noise	    =	'NO'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.050@2.00e-1,0.053@2.15e-1,0.068@2.40e-1,0.085@2.70e-1,' + \
                        '0.103@3.05e-1,0.122@3.45e-1,0.132@3.80e-1,0.290@4.30e-1,' + \
                        '0.324@4.85e-1,0.324@5.50e-1,0.296@6.25e-1,0.239@7.15e-1,' + \
                        '0.171@8.05e-1,0.294@9.20e-1,0.459@1.06e+0,0.534@1.22e+0,' + \
                        '0.573@1.41e+0,0.598@1.62e+0,0.615@1.87e+0,'               + \
                        '0.628@2.16e+0,0.636@2.40e+0,0.640@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	0.2
        noiseframe	=	1.44e5
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'rel'
    elif instr	==	'LUVOIR_ECSPEC_80_160':
        rangelo	    =	80.0
        rangehi     =   160.0
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-09
        telescope2	=	3
        telescope3	=	2
        noise	    =	'NO'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.050@2.00e-1,0.053@2.15e-1,0.068@2.40e-1,0.085@2.70e-1,' + \
                        '0.103@3.05e-1,0.122@3.45e-1,0.132@3.80e-1,0.290@4.30e-1,' + \
                        '0.324@4.85e-1,0.324@5.50e-1,0.296@6.25e-1,0.239@7.15e-1,' + \
                        '0.171@8.05e-1,0.294@9.20e-1,0.459@1.06e+0,0.534@1.22e+0,' + \
                        '0.573@1.41e+0,0.598@1.62e+0,0.615@1.87e+0,'               + \
                        '0.628@2.16e+0,0.636@2.40e+0,0.640@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	0.2
        noiseframe	=	1.44e5
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'rel'
    elif instr	==	'LUVOIR_ECSPEC_160_320':
        rangelo	    =	160.0
        rangehi     =   320.0
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-09
        telescope2	=	3
        telescope3	=	2
        noise	    =	'NO'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.050@2.00e-1,0.053@2.15e-1,0.068@2.40e-1,0.085@2.70e-1,' + \
                        '0.103@3.05e-1,0.122@3.45e-1,0.132@3.80e-1,0.290@4.30e-1,' + \
                        '0.324@4.85e-1,0.324@5.50e-1,0.296@6.25e-1,0.239@7.15e-1,' + \
                        '0.171@8.05e-1,0.294@9.20e-1,0.459@1.06e+0,0.534@1.22e+0,' + \
                        '0.573@1.41e+0,0.598@1.62e+0,0.615@1.87e+0,'               + \
                        '0.628@2.16e+0,0.636@2.40e+0,0.640@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	0.2
        noiseframe	=	1.44e5
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'rel'
    elif instr	==	'LUVOIR_ECSPEC_320_640':
        rangelo	    =	320.0
        rangehi     =   640.0
        resolution	=	1900
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1.00E-09
        telescope2	=	3
        telescope3	=	2
        noise	    =	'NO'
        noise1	    =	1
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.050@2.00e-1,0.053@2.15e-1,0.068@2.40e-1,0.085@2.70e-1,' + \
                        '0.103@3.05e-1,0.122@3.45e-1,0.132@3.80e-1,0.290@4.30e-1,' + \
                        '0.324@4.85e-1,0.324@5.50e-1,0.296@6.25e-1,0.239@7.15e-1,' + \
                        '0.171@8.05e-1,0.294@9.20e-1,0.459@1.06e+0,0.534@1.22e+0,' + \
                        '0.573@1.41e+0,0.598@1.62e+0,0.615@1.87e+0,'               + \
                        '0.628@2.16e+0,0.636@2.40e+0,0.640@2.50e+0'
        noiseoemis	=	0.1
        noisetime	=	0.2
        noiseframe	=	1.44e5
        noisepixel	=	1
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'rel'
    elif instr	==	'LUVOIR_HD':
        rangelo	    =	0.2
        rangehi     =   2.2
        resolution	=	4
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1
        telescope2	=	0
        telescope3	=	1
        noise	    =	'CCD'
        noise1	    =	2
        noise2  	=	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.250@2.00e-1,0.267@2.60e-1,0.283@2.75e-1,0.367@4.40e-1,' + \
                        '0.362@5.50e-1,0.333@6.40e-1,0.326@7.90e-1,0.702@1.26e+0,' + \
                        '0.751@1.60e+0,0.772@2.00e+0,0.780@2.20e+0'
        noiseoemis	=	0.1
        noisetime	=	3600
        noiseframe	=	1
        noisepixel	=	8
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'rel'
    elif instr	==	'LUVOIR_LUMOS':
        rangelo	    =	0.1
        rangehi     =   0.4
        resolution	=	500
        telescope	=	'SINGLE'
        diameter	=	15
        beam	    =	1
        beamunit	=	'diffrac'
        telescope1	=	1
        telescope2	=	0
        telescope3	=	1
        noise	    =	'CCD'
        noise1	    =	2
        noise2	    =	1.00E-03
        noiseotemp	=	270
        noiseoeff	=	'0.1'
        noiseoemis	=	0.1
        noisetime	=	3600
        noiseframe	=	1
        noisepixel	=	8
        transapply	=	'N'
        transshow	=	'N'
        trans	    =	'03-01'
        radunits	=	'rel'
    elif instr=='ELT-HIRES':
        rangelo    = 0.4
        rangehi    = 1.8
        resolution = 100000
        trans      = '02-00'
        transshow  = 'Y'
        transapply = 'N'
        radunits   = 'rel'
        telescope  = 'SINGLE'
        beam       = 1
        beamunit   = 'diffrac'
        diameter   = 39.3
        telescope1 = 1
        telescope2 = 0
        telescope3 = 1
        noise      = 'CCD'
        noise1     = 3
        noise2     = 1.00E-02
        noiseotemp = 283
        noiseoeff  = 0.15
        noiseoemis = 0.1
        noisetime  = 1800
        noiseframe = 1
        noisepixel = 8
    """elif instr=='':
        rangelo    =
        rangehi    =
        resolution =
        trans      =
        transshow  =
        transapply =
        radunits   =
        telescope  =
        beam       =
        beamunit   =
        diameter   =
        telescope1 =
        telescope2 =
        telescope3 =
        noise      =
        noisetime  =
        noiseframe =
        noisepixel =
        noise1     =
        noise2     =
        noiseoeff  =
        noiseoemis =
        noiseotemp =
        instrument ="""

    return rangelo, rangehi, resolution, trans, transshow, transapply,          \
           radunits, telescope, beam, beamunit, diameter,                       \
           telescope1, telescope2, telescope3,                                  \
           noise, noise1, noise2, noiseotemp, noiseoeff, noiseoemis,            \
           noisetime, noiseframe, noisepixel



In [3]:
def val(row, name, default=None):
    return row[name] if name in row.index and pd.notnull(row[name]) else default

def run_INARA_configs(row: pd.Series, *,cloudsim: bool = False)-> str:
    
    """
    Build a Frontier/INARA-style layered PSG config *from CSV values only*.
    No random sampling; if a PT parameter is missing and allow_pt_defaults=False, raises.
    """

    PIndex      = row['PlanetIndex']
    StarClass   = float(row['star_class'])
    Tstar       = float(row['star_temperature_in_Kelvin'])
    Rstar      = float(row['star_radius_in_Solar_radii'])
    sma      = float(row['semimajor_axis_of_the_planet_in_AU'])
    rplanet     = float(row['planet_radius_in_km'])
    density     = float(row['planet_density'])
    psurf       = float(row['planet_surface_pressure_bars'])
    Tsurf       = float(row['planet_surface_temperature_Kelvin'])
    dist        = float(row.get('distance_from_Earth_to_the_system_in_parsecs', 10.0))
    albedo      = float(row['planets_mean_surface_albedo'])
    avg_mw      = float(row['planet_atmospheres_avg_mol_wgt'])
    abuns       =[  float(row['H2O']),
                    float(row['CO2']),
                    float(row['O2']),
                    float(row['N2']),
                    float(row['CH4']),
                    float(row['N2O']),
                    float(row['CO']),
                    float(row['O3']),
                    float(row['SO2']),
                    float(row['NH3']), 
                    float(row['C2H6']),
                    float(row['NO2'])]
    abuns = np.array(abuns)

    # Calculate stellar luminosity
    Lstar = const.sigma_sb.value * \
            4. * np.pi * (Rstar * const.R_sun.value)**2 * Tstar**4

    mplanet = density *  (4./3. * np.pi * (rplanet * const.R_earth.cgs.value)**3) / const.M_earth.cgs.value
  
    surfgrav = const.G.cgs.value * mplanet * const.M_earth.cgs.value /  \
                   (rplanet * const.R_earth.cgs.value)**2


    # Calculate surface escape velocity [km/s^2]
    vesc = (2. * const.G.value * mplanet * const.M_earth.value / \
                (rplanet * const.R_earth.value))**0.5     / 1000.
    
    # Calculte planet's insolation Insol, relative to Earth
    Insol = Lstar / const.L_sun.value / sma**2

    # Rough approx of 'cosmic shoreline' eqn in Zahnle & Catling 2016
    # 1e-6 Insol : 0.2 vesc, 1e4 Insol : 70 vesc -- straight line on loglog
    # Slope of cosmic shoreline
    shoreline = np.log10(1e4 / 1e-6) / np.log10(70. / 0.2)

    # Calculate shoreline insolation, pinsol, value for the planet's vesc 
    # to see if it can hold an atmosphere
    # Requires the actual insolation, Insol, to be less than pinsol
    pinsol = 1e4 * (vesc / 70.)**shoreline
    if Insol < pinsol:
        canholdatmo = True


    # Calculate mean molecular weight in g/mol
    avgweight = np.sum(molweights * abuns)

    # PT parameters from CSV (no randomness)
    log10kappa   = float(row['kappa'])
    log10gamma1  = float(row['gamma1'])
    log10gamma2  = float(row['gamma2'])
    alpha   = float(row['alpha'])
    beta    = float(row['beta'])
    params_vec = np.array([log10kappa, log10gamma1, log10gamma2, alpha, beta], dtype=float)

    nlayers = 50
    pressure = np.logspace(np.log10(psurf), np.log10(1e-6), nlayers)

    temp = PT_line(
        psurf,                              # pressures in bar
        params_vec,                         # PT vector from CSV
        Rstar * const.R_sun.value,          # stellar radius [m]
        Tstar,                              # stellar Teff [K]
        float(row.get('Tint_K', 0.0)),      # internal T if provided, else 0
        sma * const.au.value,               # a [m]
        surfgrav                            # gravity [cm/s^2]
    )

    # Put temperature and pressure into array, return
    PTprof      = np.zeros((nlayers, 2), dtype=float)
    PTprof[:,0] = pressure
    PTprof[:,1] = temp


    # Condense out material
    icloud = [0, 8] #indices of cloud-forming species, H2O and NH3
    clouds = makeclouds(abuns[icloud], PTprof)

    # Repeat abuns for every layer, for easy broadcasting of cloud abundances
    layabuns = np.broadcast_to(abuns, (PTprof.shape[0], len(abuns))).copy()


    # Clouds & hazes
    if np.any(clouds[:,0]!=0) and np.any(clouds[:,1]!=0): #Both water and ammonia clouds
        layabuns[:,icloud] -= clouds #Remove H2O and NH3 gas
        naero = 2
        aeros = 'Water,Ammonia'
        atype = 'AFCRL_WATER_HRI[0.20um-0.03m],Ice_Martonchik_GSFC[0.19-200.00um]'
        aabun = '1,1'
        aunit = 'scl,scl'
        asize = '10,10' #um
        nmax = 8        #Scattering params for clouds
        lmax = 8
    elif np.any(clouds[:,0]!=0): #Only H2O clouds
        clouds = clouds[:,0]
        layabuns[:,0] -= clouds #Remove H2O gas
        naero = 1
        aeros = 'Water'
        atype = 'AFCRL_WATER_HRI[0.20um-0.03m]'
        aabun = '1'
        aunit = 'scl'
        asize = '10'    #um
        nmax = 8        #Scattering params for clouds
        lmax = 8
    elif np.any(clouds[:,1]!=0): #Only NH3 clouds
        clouds = clouds[:,1]
        layabuns[:,1] -= clouds #Remove NH3 gas
        naero = 1
        aeros = 'Ammonia'
        atype = 'Ice_Martonchik_GSFC[0.19-200.00um]'
        aabun = '1'
        aunit = 'scl'
        asize = '10'    #um
        nmax = 8        #Scattering params for clouds
        lmax = 8
    else: #No clouds
        clouds = None
        naero  = 0
        aeros  = ''
        atype  = ''
        aabun  = ''
        aunit  = ''
        asize  = ''
        nmax   = 0
        lmax   = 0

    StarType = ''
    ObjSeason = 90
    GeoPhase = 90
    GeoSolAng = 77.750
    GeoStarFrac = 0
    if StarClass == 3:
        StarType = 'U'
    elif StarClass == 4:
        StarType = 'G'
    elif StarClass == 5:
        StarType = 'K'
    elif StarClass == 6:
        StarType = 'M'
        ObjSeason = 180
    else:
        print("Error packaging starclass into params.\n")

   
    with open("default.config", "r", encoding="utf-8") as f:
        defaultConfig = f.read()
    
    config = config_str_to_dict(defaultConfig)

    # Replace values in config
    # System information
    config['OBJECT-NAME']               = "INARA-"+str(PIndex)          # Planet name
    config['OBJECT-DIAMETER']           = 2.0*rplanet                   # Planet diameter [km]
    config['OBJECT-GRAVITY']            = density                       # Planet density [g/cm3]
    config['OBJECT-STAR-DISTANCE']      = dist                       # Semimajor axis [AU]
    config['OBJECT-STAR-TYPE']          = StarType                      # Stellar class
    config['OBJECT-STAR-TEMPERATURE']   = Tstar                         # Stellar temperature [K]
    config['OBJECT-STAR-RADIUS']        = Rstar                        # Stellar radius [Rsun]
    config['GEOMETRY-OBS-ALTITUDE']     = dist                          # Distance to system
    if StarType=='M':                                                   # System geometry
        config['OBJECT-SEASON'] = 180.0             # Transit
    else:
        config['OBJECT-SEASON']          = 90.0     # Greatest separation for direct obs
        config['GEOMETRY-PHASE']         = 90.0
        config['GEOMETRY-SOLAR-ANGLE']   = 77.750
        config['GEOMETRY-STAR-FRACTION'] = 0
    # Atmosphere params
    config['ATMOSPHERE-PRESSURE'] = psurf        # Planetary surface pressure [bars]
    config['ATMOSPHERE-WEIGHT']   = avg_mw       # Molecular weight of atmosphere [g/mol]
    config['ATMOSPHERE-NGAS']     = len(MOLECULES)      # Number of gases in atmosphere
    config['ATMOSPHERE-GAS']  = ','.join(MOLECULES)      # Gases in atmosphere
    config['ATMOSPHERE-TYPE'] = ','.join([HITDICT[i] for i in MOLECULES])    # HITRAN line lists
    config['ATMOSPHERE-ABUN'] = '1,'*(len(layabuns[0])-1)+ '1'        # Gas abundances
    config['ATMOSPHERE-UNIT'] = 'scl,'*(len(layabuns[0])-1)+ 'scl'    # Unit for abundances
    config['ATMOSPHERE-LAYERS-MOLECULES'] = ','.join(MOLECULES)         # Molecules


    
    for i in range(len(PTprof)):                    # Atmosphere model layer info
        config['ATMOSPHERE-LAYER-'+str(i+1)] = ','.join(map(str, PTprof[i])) + \
                                         ',' + ','.join(map(str, list(layabuns[i].astype(str))))
        if cloudsim==True:
            if np.all(clouds != None):                                  # Add cloud info
                if i==0:
                    config['ATMOSPHERE-LAYERS-MOLECULES'] += ',' + aeros    # Cloud names
                if len(clouds.shape) > 1:                               # Cloud abundances
                    config['ATMOSPHERE-LAYER-'+str(i+1)] += ',' + \
                                               ','.join(map(str, list(clouds[i].astype(str))))
                else:
                    config['ATMOSPHERE-LAYER-'+str(i+1)] += ',' + str(clouds[i])
    # Surface info
    config['SURFACE-TEMPERATURE'] = Tsurf
    config['SURFACE-ALBEDO']      = albedo
    config['SURFACE-EMISSIVITY']  = 1. - albedo
    # Cloud/aerosol info
    if cloudsim==True:
        config['ATMOSPHERE-NAERO'] = naero
        config['ATMOSPHERE-AEROS'] = aeros
        config['ATMOSPHERE-ATYPE'] = atype
        config['ATMOSPHERE-AABUN'] = aabun
        config['ATMOSPHERE-AUNIT'] = aunit
        config['ATMOSPHERE-ASIZE'] = asize
        config['ATMOSPHERE-NMAX']  = nmax
        config['ATMOSPHERE-LMAX']  = lmax

    # Set observation params based on system type, and necessary resolution for clouds
    if cloudsim==False:
        noisesims = 6
        if StarType=='M':  #Transits
            observations = ['LUVOIR_ECSPEC_02_03',   'LUVOIR_ECSPEC_03_04',  \
                            'LUVOIR_ECSPEC_04_06',   'LUVOIR_ECSPEC_06_08',  \
                            'LUVOIR_ECSPEC_08_13',   'LUVOIR_ECSPEC_13_2',   \
                            'LUVOIR_ECSPEC_2_4',                             \
                            'LUVOIR_ECSPEC_4_8',     'LUVOIR_ECSPEC_8_12',   \
                            'LUVOIR_ECSPEC_12_20',   'LUVOIR_ECSPEC_20_40',  \
                            'LUVOIR_ECSPEC_40_80',   'LUVOIR_ECSPEC_80_160', \
                            'LUVOIR_ECSPEC_160_320', 'LUVOIR_ECSPEC_320_640']
        else:               #Direct observations
            observations = ['LUVOIR_ECLIPS_02_03',   'LUVOIR_ECLIPS_03_04',  \
                            'LUVOIR_ECLIPS_04_06',   'LUVOIR_ECLIPS_06_08',  \
                            'LUVOIR_ECLIPS_08_13',   'LUVOIR_ECLIPS_13_2',   \
                            'LUVOIR_ECLIPS_2_4',                             \
                            'LUVOIR_ECLIPS_4_8',     'LUVOIR_ECLIPS_8_12',   \
                            'LUVOIR_ECLIPS_12_20',   'LUVOIR_ECLIPS_20_40',  \
                            'LUVOIR_ECLIPS_40_80',   'LUVOIR_ECLIPS_80_160', \
                            'LUVOIR_ECLIPS_160_320', 'LUVOIR_ECLIPS_320_640']
    elif cloudsim==True:
        # These need to be updated for the cloud case (smaller intervals)
        noisesims = 6
        if StarType=='M':  #Transits
            observations = ['LUVOIR_ECSPEC_02_03',   'LUVOIR_ECSPEC_03_04',  \
                            'LUVOIR_ECSPEC_04_06',   'LUVOIR_ECSPEC_06_08',  \
                            'LUVOIR_ECSPEC_08_13',   'LUVOIR_ECSPEC_13_2',   \
                            'LUVOIR_ECSPEC_2_4',                             \
                            'LUVOIR_ECSPEC_4_8',     'LUVOIR_ECSPEC_8_12',   \
                            'LUVOIR_ECSPEC_12_20',   'LUVOIR_ECSPEC_20_40',  \
                            'LUVOIR_ECSPEC_40_80',   'LUVOIR_ECSPEC_80_160', \
                            'LUVOIR_ECSPEC_160_320', 'LUVOIR_ECSPEC_320_640']
        else:               #Direct observations
            observations = ['LUVOIR_ECLIPS_02_03',   'LUVOIR_ECLIPS_03_04',  \
                            'LUVOIR_ECLIPS_04_06',   'LUVOIR_ECLIPS_06_08',  \
                            'LUVOIR_ECLIPS_08_13',   'LUVOIR_ECLIPS_13_2',   \
                            'LUVOIR_ECLIPS_2_4',                             \
                            'LUVOIR_ECLIPS_4_8',     'LUVOIR_ECLIPS_8_12',   \
                            'LUVOIR_ECLIPS_12_20',   'LUVOIR_ECLIPS_20_40',  \
                            'LUVOIR_ECLIPS_40_80',   'LUVOIR_ECLIPS_80_160', \
                            'LUVOIR_ECLIPS_160_320', 'LUVOIR_ECLIPS_320_640']
    else:
        print("Invalid specification for cloudsim. Only use True or False.")
        sys.exit()


    # Perform each observation, and store the results and config
    allresults = []
    allconfigs = []

    for i in range(len(observations)):
        rangelo, rangehi, resolution, trans, transshow, transapply,              \
            radunits, telescope, beam, beamunit, diameter,                       \
            telescope1, telescope2, telescope3,                                  \
            noise, noise1, noise2, noiseotemp, noiseoeff, noiseoemis,            \
            noisetime, noiseframe, noisepixel = setobsparams(observations[i])

        # For transit observations, scale exposure/frames based on distance
        if StarType=='M':
            noisetime  = (dist/12.1)**2  * noisetime
        else:
            noisetime  = (dist/ 5.0)**2  * noisetime

        # Fill in config
        config['GENERATOR-RANGE1']      = rangelo
        config['GENERATOR-RANGE2']      = rangehi
        config['GENERATOR-RESOLUTION']  = resolution
        config['GENERATOR-TRANS']       = trans
        config['GENERATOR-TRANS-SHOW']  = transshow
        config['GENERATOR-TRANS-APPLY'] = transapply
        config['GENERATOR-RADUNITS']    = radunits
        config['GENERATOR-TELESCOPE']   = telescope
        config['GENERATOR-BEAM']        = beam
        config['GENERATOR-BEAM-UNIT']   = beamunit
        config['GENERATOR-DIAMTELE']    = diameter
        config['GENERATOR-TELESCOPE1']  = telescope1
        config['GENERATOR-TELESCOPE2']  = telescope2
        config['GENERATOR-TELESCOPE3']  = telescope3
        config['GENERATOR-NOISE']       = noise
        config['GENERATOR-NOISE1']      = noise1
        config['GENERATOR-NOISE2']      = noise2
        config['GENERATOR-NOISEOTEMP']  = noiseotemp
        config['GENERATOR-NOISEOEFF']   = noiseoeff
        config['GENERATOR-NOISEOEMIS']  = noiseoemis
        config['GENERATOR-NOISETIME']   = noisetime
        config['GENERATOR-NOISEFRAMES'] = noiseframe
        config['GENERATOR-NOISEPIXELS'] = noisepixel

        # Store config
        allconfigs.append(config.copy())

        result = post_psg_config(PSG_URL, config_dict_to_str(config))
        allresults.append(result)

        print("observation "+observations[i]+" complete ")


    # Package the results, concatenate the configs
    nvals    = [allresults[i]['spectrum'].shape[0] for i in range(len(allresults))]
    results  = np.zeros((np.sum(nvals), 5), dtype=np.float64)
    if StarType=='M':
        inonoise = [0, 1, 3, 4]
    else:
        inonoise = [0, 1,    4]
    #inonoise = [0, 1, 3, 4]

    for i in range(len(allresults)):
        if i < noisesims:
            results[int(np.sum(nvals[:i])) : int(np.sum(nvals[:i]))+nvals[i]] = \
                         allresults[i]['spectrum'][:,:results.shape[1]]
        else:
            try:
                inonoise = [0, 1, 3, 4]
                results[int(np.sum(nvals[:i])) : int(np.sum(nvals[:i]))+nvals[i], inonoise] = \
                         allresults[i]['spectrum'][:,:results.shape[1]-1]
            except:
                inonoise = [0, 1, 4]
                results[int(np.sum(nvals[:i])) : int(np.sum(nvals[:i]))+nvals[i], inonoise] = \
                         allresults[i]['spectrum'][:,:results.shape[1]-2]

    configstrs = [config_dict_to_str(allconfigs[i]) for i in range(len(allconfigs))]
    configs    = '\n\n'.join(configstrs)

    return results, configs

# Sensitivity of PSG OLR to T

In [ ]:
df_sens = pd.read_csv('psg_models.csv')
#T_original = df_sens[0, 'planet_surface_temperature_Kelvin']
T_original = df_sens['planet_surface_temperature_Kelvin']

base_row = df_sens.iloc[0].copy()

T = np.linspace(T_original - 100, T_original + 100, 20)  # 20 points

# replicate the base row 20 times
df_sens = pd.DataFrame([base_row] * len(T)).reset_index(drop=True)

df_sens['planet_surface_temperature_Kelvin'] = T

out_dir = Path(r"C:\Users\Andrew\OneDrive\Planetary heat flow models\Models\INARA PSG Generator\Sensitivity Points")

df_sens




,PlanetIndex,star_class,star_temperature_in_Kelvin,star_radius_in_Solar_radii,distance_from_Earth_to_the_system_in_parsecs,semimajor_axis_of_the_planet_in_AU,planet_radius_in_km,planet_density,planet_surface_pressure_bars,kappa,...,CH4,N2O,CO,O3,SO2,NH3,C2H6,NO2,planet_atmospheres_avg_mol_wgt,planets_mean_surface_albedo
0,0.0,4.0,5772.0,1.0,1.0,1.0,6378.0,5.5,1.01325,-2.729805,...,0.000002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.25721,0.5
1,0.0,4.0,5772.0,1.0,1.0,1.0,6378.0,5.5,1.01325,-2.729805,...,0.000002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.25721,0.5
2,0.0,4.0,5772.0,1.0,1.0,1.0,6378.0,5.5,1.01325,-2.729805,...,0.000002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.25721,0.5
3,0.0,4.0,5772.0,1.0,1.0,1.0,6378.0,5.5,1.01325,-2.729805,...,0.000002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.25721,0.5
4,0.0,4.0,5772.0,1.0,1.0,1.0,6378.0,5.5,1.01325,-2.729805,...,0.000002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.25721,0.5
5,0.0,4.0,5772.0,1.0,1.0,1.0,6378.0,5.5,1.01325,-2.729805,...,0.000002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.25721,0.5
6,0.0,4.0,5772.0,1.0,1.0,1.0,6378.0,5.5,1.01325,-2.729805,...,0.000002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.25721,0.5
7,0.0,4.0,5772.0,1.0,1.0,1.0,6378.0,5.5,1.01325,-2.729805,...,0.000002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.25721,0.5
8,0.0,4.0,5772.0,1.0,1.0,1.0,6378.0,5.5,1.01325,-2.729805,...,0.000002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.25721,0.5
9,0.0,4.0,5772.0,1.0,1.0,1.0,6378.0,5.5,1.01325,-2.729805,...,0.000002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.25721,0.5


In [ ]:
INARA_INPUT_FILE = r"C:\Users\andrew\OneDrive - University of Cape Town\Documents\Work\4th year\CHE4045Z - Research\Andrew Stoffels's files - Planetary heat flow models\Models\INARA PSG Generator\Sensitivity Points\PSG configurations"
INARA_CONFIG_DIR = r"C:\Users\andrew\OneDrive - University of Cape Town\Documents\Work\4th year\CHE4045Z - Research\Andrew Stoffels's files - Planetary heat flow models\Models\INARA PSG Generator\INARA Configs"

In [ ]:
input_df = pd.read_csv(INARA_INPUT_FILE)
indices = list(range(len(input_df)))

outdir = Path(INARA_CONFIG_DIR)
outdir.mkdir(parents=True, exist_ok=True)

for i in indices:
        if i < 0 or i >= len(input_df):
            print(f"[skip] index {i} out of range (0..{len(input_df)-1})", file=sys.stderr)
            continue
        row = input_df.iloc[i]
        results, configs = run_INARA_configs(row, cloudsim=False)
        df = pd.DataFrame(results, columns=['wavelength_(um)','star_planet_signal_(erg/s/cm2)','noise_(erg/s/cm2)','stellar_signal_(erg/s/cm2)','planet_signal_(erg/s/cm2)'])
        df.to_csv('C:\\Users\\andrew\\OneDrive - University of Cape Town\\Documents\\Work\\4th year\\CHE4045Z - Research\\Andrew Stoffels\'s files - Planetary heat flow models\\Models\\INARA PSG Generator\\INARA Configs\\psg_config_row'+str(i)+'.csv',index=False)


C:\Users\henry\AppData\Local\Temp\ipykernel_15136\567715857.py:357: RuntimeWarning: invalid value encountered in divide
  cloudsnh3[inh3] = abun[1] * ((abun[1] * PTprof[:,0])[inh3] - laysvp[inh3,1]) / \


observation LUVOIR_ECLIPS_02_03 complete 
observation LUVOIR_ECLIPS_03_04 complete 
observation LUVOIR_ECLIPS_04_06 complete 
observation LUVOIR_ECLIPS_06_08 complete 
observation LUVOIR_ECLIPS_08_13 complete 
observation LUVOIR_ECLIPS_13_2 complete 
observation LUVOIR_ECLIPS_2_4 complete 
observation LUVOIR_ECLIPS_4_8 complete 
observation LUVOIR_ECLIPS_8_12 complete 
observation LUVOIR_ECLIPS_12_20 complete 
observation LUVOIR_ECLIPS_20_40 complete 
observation LUVOIR_ECLIPS_40_80 complete 
observation LUVOIR_ECLIPS_80_160 complete 
observation LUVOIR_ECLIPS_160_320 complete 
observation LUVOIR_ECLIPS_320_640 complete 
Error packaging starclass into params.

observation LUVOIR_ECLIPS_02_03 complete 
observation LUVOIR_ECLIPS_03_04 complete 
observation LUVOIR_ECLIPS_04_06 complete 
observation LUVOIR_ECLIPS_06_08 complete 
observation LUVOIR_ECLIPS_08_13 complete 
observation LUVOIR_ECLIPS_13_2 complete 
observation LUVOIR_ECLIPS_2_4 complete 
observation LUVOIR_ECLIPS_4_8 complete 
